# Credit Default Prediction Jupyter Notebook

This Jupyter Notebook covers the following topics and can be used to test and compare 3 implemented models: 

1. Data Loading & Merging
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Preprocessing
5. Model Training includes Random Forest, Logistic Regression, XGBoost
6. Model Comparison

**End Goal:** Predicting whether an applicant will default on their credit card using application data and credit history.
Dataset Source: https://www.kaggle.com/datasets/rikdifos/credit-card-approval-prediction?resource=download



## Setup


In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isdir(os.path.join(project_root, "src")):

    project_root = os.getcwd()
os.chdir(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Working directory: {os.getcwd()}")

In [ ]:
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

print("All libraries imported successfully.")

## Step 1 — Data Loading & Merging

Two raw CSV files are merged into a single dataset with one row per applicant:
- application_record.csv: applicant features (income, employment, family status, etc.)
- credit_record.csv: monthly payment history

The target column default is set to 1 if the applicant was ever 60+ days overdue (STATUS 2–5), otherwise it is an 0.

In [ ]:
from src.data.make_dataset import CreditDatasetBuilder

builder = CreditDatasetBuilder()
merged_df = builder.build()

In [ ]:
print(f"Merged dataset shape: {merged_df.shape}")
print(f"\nColumn list ({len(merged_df.columns)} columns):")

print(merged_df.columns.tolist())

print("\nFirst 5 rows:")
merged_df.head()

In [ ]:
print("Data types:")
print(merged_df.dtypes.value_counts())

print(f"\nMissing values: {merged_df.isnull().sum().sum()} total")

if merged_df.isnull().sum().sum() > 0:
    print(merged_df.isnull().sum()[merged_df.isnull().sum() > 0])

---
## Step 2 — Exploratory Data Analysis (EDA)

Before modelling, the data needs to be understood. This includes knowing: 

- If the target class is imbalanced?
- How are the features distributed? Are there any outliers?
- Are there any features that are highly correlated?


### 2.1 Target Variable Distribution

Through EDA we can see if the dataset is heavily imbalanced which means most applicants do not default. This is where the use of SMOTE and `class_weight='balanced'` is needed in the models.

In [ ]:
counts = merged_df["default"].value_counts().sort_index()
total = len(merged_df)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(["No Default", "Default"], counts.values, color=["steelblue", "tomato"], width=0.5)
ax.set_title("Target Variable Distribution")
ax.set_ylabel("Count")

#count and percentage labels on each bar
labels = ["No Default", "Default"]

for i in range(len(counts.values)):
    v = counts.values[i]
    ax.text(i, v + 50, f"{v:,} ({v/total*100:.1f}%)", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

print(f"No Default: {counts[0]:,} ({counts[0]/total*100:.2f}%)")
print(f"Default:    {counts[1]:,} ({counts[1]/total*100:.2f}%)")
print(f"Imbalance ratio: {counts[0]/counts[1]:.1f}:1")

**Interpretation:** This shows us that the data is heavily imbalanced at just 1.7% of applicants being in the default category, with no defaulters being 98.31% of the data. The imbalance ratio is 58.2 which means that for every 1 defaulter there are 58.2 non defaulters. What this means is that we need SMOTE and scale_pos_weight to balance the data otherwise the model would just predict "No Default" for everyone with a 98% accuracy.

### 2.2 Summary Statistics for Numerical Features

In [ ]:

#separate numerical and categorical features as well as ID, target and status columns 
skip_cols = {"ID", "default", "status_0", "status_1", "status_2", "status_3", "status_4", "status_5", "status_C", "status_X", "FLAG_MOBIL"}

num_cols = []
for c in merged_df.select_dtypes(include="number").columns:

    if c not in skip_cols:
        num_cols.append(c)

cat_cols = []
for c in merged_df.select_dtypes(include="object").columns:
    
    if c not in skip_cols:
        cat_cols.append(c)

print(f"Numerical features: {num_cols}")
print(f"Categorical features: {cat_cols}")

merged_df[num_cols].describe().round(2)

**Interpretation:**
- DAYS_BIRTH and DAYS_EMPLOYED is an negative integers in the dataset. What this means is that these need to be converted in feature engineering to age_years and years_employed
- DAYS_EMPLOYED has a placeholder value of 365243 for unemployed/retired applicants so it needs to be replaced with 0.
- AMT_INCOME_TOTAL has high variance with minimum value of $27,000.00 all the way to an maximum value of $1,575,000.00

### 2.3 Histograms of Numerical Features


In [ ]:
n_rows = (len(num_cols) + 2) // 3

#histograms for each of the numerical features 
fig, axes = plt.subplots(n_rows, 3, figsize=(15, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(num_cols):

    axes[i].hist(merged_df[col].dropna(), bins=40, color="blue", edgecolor="white")
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel("Count", fontsize=8)

for j in range(i + 1, len(axes)):

    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

**Interpretation:** Most features are right-skewed. DAYS_EMPLOYED is showing  a clear jump at 365243 confirming that it is an placeholder. Income has high variance with some people with very high income pulling the distribution right.

### 2.4 Categorical Feature Bar Charts

In [ ]:

#bar chart
n_rows = (len(cat_cols) + 1) // 2
fig, axes = plt.subplots(n_rows, 2, figsize=(14, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    
    counts_cat = merged_df[col].value_counts()
    axes[i].bar(counts_cat.index, counts_cat.values, color="blue", edgecolor="white")
    axes[i].set_title(col, fontsize=11)
    axes[i].set_ylabel("Count", fontsize=9)
    axes[i].tick_params(axis="x", rotation=30, labelsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

**Interpretation:** This shows the categories that will influence the model the most since they make up the majority of the training data. 

### 2.5 Correlation Heatmap

In [ ]:
heatmap_cols = num_cols + ["default"]

#correlation matrix even the target variable (default)
corr = merged_df[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, ax=ax, annot_kws={"size": 7})

ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

**Interpretation:** Features with correlation > 0.9 are dropped in feature engineering. This shows that CNT_CHILDREN and CNT_FAM_MEMBERS are highly correlated at 0.89 which means one of these will be dropped in feature engineering. Most of the features have a low correlation with 'default' which shows that the relationship is non linear. 

### 2.6 Feature Distributions between Defaulters and Non-Defaulters

In [ ]:
boxplot_feats = []

# boxplots split by default/non default status
for c in num_cols:

    if c not in {"months_count", "months_min", "months_max"}:
        boxplot_feats.append(c)

boxplot_feats = boxplot_feats[:8]

n_rows = (len(boxplot_feats) + 1) // 2
fig, axes = plt.subplots(n_rows, 2, figsize=(13, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(boxplot_feats):

    # split into defaulters and non-defaulters
    data_groups = []
    for v in [0, 1]:
        data_groups.append(merged_df[merged_df["default"] == v][col].dropna())

    bp = axes[i].boxplot(data_groups, labels=["No Default", "Default"], patch_artist=True)

    bp["boxes"][0].set_facecolor("blue")
    bp["boxes"][1].set_facecolor("red")
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel(col, fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

**Interpretation:** This shows that most of the features look similar between defaulters and non-defaulters which means that there is not an single feature that can strongly seperate the 2 classes. This means that the model will need to learn combinations of features to predict.

---
## Step 3 — Feature Engineering

- age_years from DAYS_BIRTH
- years_employed from DAYS_EMPLOYED (365243 replaced with 0 for unemployed)
- income_per_member = income divided by family size
- income_per_child = income divided by number of children

Features that have low variance or high correlation are dropped here as mentioned before. 

In [ ]:
from src.feature_engineering.build_features import FeatureEngineer

engineer = FeatureEngineer()
engineer.run()

features_df = pd.read_csv("src/data/processed/features.csv")
print(f"\nFeature-engineered dataset shape: {features_df.shape}")

print(f"Columns: {features_df.columns.tolist()}")

In [ ]:

# top 20 features by correlation  
drop = ["ID", "default"]
X_fe = features_df.drop(columns=drop)
corr_with_target = X_fe.corrwith(features_df["default"]).abs().sort_values(ascending=False)

top20 = corr_with_target.head(20)

fig, ax = plt.subplots(figsize=(9, 6))
top20[::-1].plot(kind="barh", ax=ax, color="blue", edgecolor="white")
ax.set_xlabel("Correlation with Target")
ax.set_title("Top 20 Features by Correlation with Default")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

#rank features by mutual information 
y_fe = features_df["default"]
selector = SelectKBest(mutual_info_classif, k=min(15, X_fe.shape[1]))
selector.fit(X_fe, y_fe)

mi_scores = pd.DataFrame({
    "feature": X_fe.columns,
    "mutual_info": selector.scores_
}).sort_values("mutual_info", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
mi_scores[::-1].plot(x="feature", y="mutual_info", kind="barh", ax=ax,
                     color="blue", edgecolor="white", legend=False)
ax.set_xlabel("Mutual Information Score")
ax.set_title("Top 15 Features by Mutual Information")
plt.tight_layout()
plt.show()

print(mi_scores.to_string(index=False))

**Interpretation:** The top features are mostly monthly payment status counts and income-related columns. The demographic features like income, gender, housing have much weaker scores which means that credit history will be better at predicting in the model and we will see credit history related columns used the most in the models.

---
## Step 4 — Preprocessing

The preprocessing step handles missing values, outliers and splitting the data:

- Missing values are filled with the average for numerical columns and mode for categorical
- Outliers are capped using Interquartile range
- Categorical columns are converted to numbers
- Features are scaled for Logistic Regression
- Data is split 80/20 between training and testing to keep the same default rate in both sets of data

In [ ]:
from src.preprocessing_data.preprocessing import CreditDataPreprocessor

preprocessor = CreditDataPreprocessor(input_path="src/data/processed/features.csv")
train_df, test_df = preprocessor.run()

In [ ]:
print(f"Training set:  {train_df.shape}")
print(f"Test set:      {test_df.shape}")

print(f"\nTraining target distribution:")

print(train_df["default"].value_counts(normalize=True).round(4))

print(f"\nTest target distribution:")

print(test_df["default"].value_counts(normalize=True).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, df, title in zip(axes, [train_df, test_df], ["Training Set", "Test Set"]):
    counts_split = df["default"].value_counts().sort_index()

    ax.bar(["No Default", "Default"], counts_split.values,
           color=["blue", "red"], edgecolor="white", width=0.5)
    
    ax.set_title(title)
    ax.set_ylabel("Count")
    
    for i, v in enumerate(counts_split.values):
        ax.text(i, v + 50, f"{v:,} ({v/len(df)*100:.1f}%)", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

**Interpretation:** This shows that both sets of data now has the same default rate this confirms that the split worked.

---
## Step 5a Model: Random Forest

Random Forest builds multiple decision trees and combines their predictions. SMOTE is used to balance the training data and class_weight='balanced' gives more weight to defaulters since the data is imbalanced.

Hyperparameters tuned using RandomizedSearchCV (5-fold CV, F1 scoring):
- n_estimators: 100, 200, 300, 500
- max_depth: 10, 20, 30, None
- min_samples_split: 2, 5, 10
- min_samples_leaf: 1, 2, 4
- max_features: sqrt, log2

In [ ]:
from src.models.train_model import RandomForestModel

rf_model = RandomForestModel()

rf_metrics = rf_model.run()

In [ ]:
display(Image("reports/figures/rf_confusion_matrix.png"))

display(Image("reports/figures/rf_roc_curve.png"))

display(Image("reports/figures/rf_feature_importance.png"))

In [ ]:
print("Random Forest Test Metrics:")

for k, v in rf_metrics.items():
    print(f"  {k:>12s}: {v:.4f}")

target_met = rf_metrics["accuracy"] >= 0.80


if target_met:
    result = "MET"
    
else:
    result = "NOT MET"

print(f"\n  Accuracy (>=0.80): {result}")

**Interpretation:** The confusion matrix shows how many defaults were correctly caught vs missed by the Random Forest model where it was able to identify 7158 non defaulters and 23 actual defaulters, but it missed 100 defaults (false negatives). The ROC curve shows higher AUC of 0.883 which means the model is better at separating defaulters from non-defaulters. The feature importance plot shows which features the model used most and in this model it used months_count, status_0, and status_C the most where months_count = total months the applicant had a credit history for, status_0 = number of months they were late between 1-29 days late and status_C = number of months they fully paid off their balance which means that credit history length and repayment behaviour mattered in this model.

---
## Step 5b Model: Logistic Regression

Logistic Regression predicts the probability of default using a linear combination of features which means that this gives a baseline to compare against Random Forest model.

SMOTE is used to balance the training data and Hyperparameters tuned using GridSearchCV (5-fold CV, ROC AUC scoring):
- C: 0.001, 0.01, 0.1, 1, 10
- penalty: l1, l2
- solver: liblinear, saga
- class_weight: balanced, None

In [ ]:
from src.models.train_model import LogisticRegressionModel

lr_model = LogisticRegressionModel()
lr_metrics = lr_model.run()

In [ ]:
display(Image("reports/figures/lr_confusion_matrix.png"))

display(Image("reports/figures/lr_roc_curve.png"))


In [ ]:
print("Logistic Regression: Test Metrics:")
for k, v in lr_metrics.items():
    print(f"  {k:>12s}: {v:.4f}")

target_met = lr_metrics["accuracy"] >= 0.80


if target_met:
    result = "MET"
    
else:
    result = "NOT MET"

print(f"\n  Accuracy (>=0.80): {result}")

**Interpretation:** Logistic Regression was able to catch 88 actual defaulters but had 1369 false positives which means it flags too many non-defaulters as defaults. The AUC of 0.806 is lower than Random Forest showing the data has non-linear pattern and this model is not able to capture it effectively.

---
## Step 5c Model: XGBoost

Class imbalance is handled using scale_pos_weight which gives more weight to defaulters without oversampling and Hyperparameters are tuned using RandomizedSearchCV (5-fold CV, F1 scoring):
- n_estimators: 100, 200, 300, 500
- max_depth: 3, 5, 7, 10
- learning_rate: 0.01, 0.05, 0.1, 0.2
- subsample: 0.6, 0.8, 1.0
- colsample_bytree: 0.6, 0.8, 1.0ch.

In [ ]:
from src.models.train_model import XGBoostModel

xgb_model = XGBoostModel()
xgb_metrics = xgb_model.run()

In [ ]:
display(Image("reports/figures/model_xgb_confusion_matrix.png"))

display(Image("reports/figures/xgb_roc_curve.png"))

display(Image("reports/figures/xgb_feature_importance.png"))

In [ ]:
print("XGBoost: Test Metrics:")

for k, v in xgb_metrics.items():
    print(f"  {k:>12s}: {v:.4f}")

target_met = xgb_metrics["accuracy"] >= 0.80

if target_met:
    result = "MET"
else:
    result = "NOT MET"

print(f"\n  Accuracy (>=0.80): {result}")

**Interpretation:** XGBoost caught 58 defaulters and missed 65, with only 29 false positives which is much better than Logistic Regression. The AUC is 0.889 which is highest out of all the three models. Feature importance shows months_max and months_count as the top features, which shows that the credit history matters and is important for predicting defaulters vs non defaulters.

---
## Step 6 — Model Comparison

All models were trained on the training set only, the test data was never introduced which will give it an unbiased evaluation. The comparison of models will be done using ROC AUC because the accuracy will be misleading otherwise with imbalanced data. 

In [ ]:
from src.models.predict_model import ModelComparator

comparator = ModelComparator()
comparison_df, best_model_name = comparator.run()

In [ ]:
display(Image("reports/figures/model_comparison_bar.png"))

display(Image("reports/figures/roc_comparison.png"))

In [ ]:
print("All Model Metrics on Test Data Set:")
print(comparison_df.round(4).to_string())

print(f"\nBest model (by ROC AUC): {best_model_name}")

In [ ]:

# bar chart for comparing all 3 models across all metrics
metrics_list = ["Accuracy", "Precision", "Recall", "F1-Score", "ROC AUC"]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_list))
width = 0.25

for i, model in enumerate(comparison_df.index):
    values = comparison_df.loc[model, metrics_list].values
    ax.bar(x + i * width, values, width, label=model)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_list)
ax.set_ylim(0, 1.1)

ax.set_ylabel("Score")
ax.set_title("Model Comparison")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import joblib
from sklearn.metrics import roc_curve, roc_auc_score

test_data = pd.read_csv("src/data/processed/test.csv")

target_col = "default"

X_test_eval = []
for c in test_data.columns:
    if c != target_col:
        X_test_eval.append(c)

X_test_eval = test_data[X_test_eval].values
y_test_eval = test_data[target_col].values

# model paths 
model_files = {
    "Logistic Regression": "src/models/saved/logistic_regression_best.joblib",

    "Random Forest": "src/models/saved/random_forest_best.joblib",
    
    "XGBoost": "src/models/saved/xgboost_model.joblib",
}

fig, ax = plt.subplots(figsize=(7, 6))

# plotting ROC curve for each model
for name, path in model_files.items():
    if os.path.exists(path):
        m = joblib.load(path)
        y_prob = m.predict_proba(X_test_eval)[:, 1]
        fpr, tpr, _ = roc_curve(y_test_eval, y_prob)
        auc = roc_auc_score(y_test_eval, y_prob)
        ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc:.3f})")

ax.plot([0, 1], [0, 1], color="grey", linestyle="--", lw=1)

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")

ax.set_title("ROC Curve Comparison")

ax.legend(loc="lower right")
plt.tight_layout()
plt.show()